# Demo: RetinaFace MNV1 0.25 เทียบ MTCNN — วาดทับภาพเดียวกัน

ใช้ **TFLite FP16 ที่เลือกหลังเทรน** ทั้งสองโมเดล Input และภาพต้นฉบับชุดเดียวกัน

- **MTCNN: ฟ้า เส้นทึบ จุดวงกลมโปร่ง ○**
- **RetinaFace MNV1 0.25: ส้ม เส้นประ จุดกากบาท ×**
- ภาพแรก: ต้นฉบับและผลซ้อนทับ จากนั้นซูมทุกหน้าที่อย่างน้อยหนึ่งโมเดลตรวจพบ
- เลข landmark: **1 ตาฝั่งซ้ายภาพ / 2 ตาฝั่งขวาภาพ / 3 จมูก / 4 มุมปากฝั่งซ้ายภาพ / 5 มุมปากฝั่งขวาภาพ**

ดูว่าจุดตรงกับตา จมูก และมุมปากมากแค่ไหนได้ด้วยสายตา แต่ภาพ demo ไม่มี ground truth จึงยังตัดสินความแม่นยำเชิงตัวเลขไม่ได้ และ confidence ข้ามโมเดลไม่ใช่คะแนนความแม่นยำที่เทียบตรงกันได้

ใส่ภาพใน `images/` แล้ว Run All ได้เลย ไม่มี PyTorch ไม่ต้องเทรนเพิ่ม


ผลที่บันทึกไว้เป็นการตรวจแพ็กบน Windows CPU ด้วยภาพ WIDER FACE 3 ภาพ (portrait/group/crowd); ไม่รวมไฟล์ภาพต้นฉบับใน ZIP เมื่อรันบน Mac ให้ใส่ภาพของคุณใน images/ โค้ดตั้งค่ากลับเป็นเส้นทางภายในแพ็กแล้ว

In [ ]:
from pathlib import Path
import sys
import json
import csv
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'model_manifest.json').exists():
    ROOT = ROOT.parent
assert (ROOT / 'model_manifest.json').exists(), 'Open this notebook inside the extracted package.'
sys.path.insert(0, str(ROOT))

from inference import Detector
from inference import load_image

INPUT = ROOT / 'images'
OUTPUT = ROOT / 'outputs' / ('overlay_' + time.strftime('%Y%m%d_%H%M%S'))
SIZE = (480, 640)  # Width, height. Portrait only in this team package.
CONFIDENCE = 0.5
THREADS = 2
PAIR_IOU = 0.3  # Only groups predictions for zooming; NOT a correctness metric.

STYLES = {
    'mtcnn': {'name': 'MTCNN', 'short': 'M', 'color': '#00aadd', 'line': '-', 'marker': 'o'},
    'retinaface_mnv1_025': {'name': 'RetinaFace MNV1 0.25', 'short': 'R',
                          'color': '#ff8800', 'line': '--', 'marker': 'x'},
}


## 1. รันทั้งสองโมเดลกับภาพชุดเดียวกัน
เก็บพิกัดบนภาพต้นฉบับหลังจัดทิศ EXIF ทุกภาพ ไม่ใช้ผลรอบเก่าและไม่ตัดเหลือเฉพาะหน้าที่ทั้งคู่พบ


In [ ]:
extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
if INPUT.is_file():
    paths = [INPUT]
    input_root = INPUT.parent
else:
    paths = sorted(path for path in INPUT.rglob('*')
                   if path.is_file() and path.suffix.lower() in extensions
                   and not path.resolve().is_relative_to(OUTPUT.resolve()))
    input_root = INPUT
if not paths:
    raise ValueError('Put JPG/PNG images in images/ or change INPUT to your image folder.')
OUTPUT.mkdir(parents=True, exist_ok=False)
detectors = {}
for name in STYLES:
    detectors[name] = Detector(name, SIZE, THREADS, CONFIDENCE)

results = []
summary = []
for image_index, path in enumerate(paths, 1):
    image = load_image(path)
    relative = path.relative_to(input_root).as_posix()
    predictions = {}
    for name, detector in detectors.items():
        boxes, points = detector(image)
        predictions[name] = {'boxes': boxes, 'points': points}
        summary.append({'image': relative, 'model': name, 'faces': len(boxes),
                        'inference_ms': detector.last_ms})
    result = {'path': path, 'image': relative, 'predictions': predictions}
    results.append(result)
    serialized = {'image': relative, 'input_wh': list(SIZE),
                  'coordinate_space': 'EXIF-oriented original image pixels', 'models': {}}
    for name, prediction in predictions.items():
        serialized['models'][name] = {
            'boxes_xyxy_score': prediction['boxes'].tolist(),
            'landmarks_xy': prediction['points'].tolist(),
        }
    (OUTPUT / f'{image_index:04d}_predictions.json').write_text(
        json.dumps(serialized, indent=2), encoding='utf-8'
    )
    print(f"[{image_index}/{len(paths)}] {relative}: "
          f"MTCNN {len(predictions['mtcnn']['boxes'])} faces | "
          f"RetinaFace {len(predictions['retinaface_mnv1_025']['boxes'])} faces")

with (OUTPUT / 'summary.csv').open('w', newline='', encoding='utf-8') as stream:
    writer = csv.DictWriter(stream, fieldnames=['image', 'model', 'faces', 'inference_ms'])
    writer.writeheader()
    writer.writerows(summary)


## 2. ฟังก์ชันวาดและจัดกลุ่มภาพซูม
จับคู่กรอบด้วย IoU แบบหนึ่งต่อหนึ่งเพื่อซูมคู่เดียวกันเท่านั้น ไม่ลบกรอบของโมเดลใด และแสดงกรอบที่ไม่มีคู่ด้วย
กรอบซูมเป็นบริเวณดูภาพ ไม่ใช่ ground truth หากสองโมเดลวางกรอบต่างกันมากอาจแสดงเป็นคนละภาพซูม


In [ ]:
def legend_handles():
    handles = []
    for style in STYLES.values():
        handles.append(Line2D([0], [0], color=style['color'], linestyle=style['line'],
                              marker=style['marker'], markerfacecolor='none', markersize=8,
                              label=style['name'], linewidth=2))
    return handles


def draw_predictions(axis, predictions, numbered=False):
    for name, prediction in predictions.items():
        style = STYLES[name]
        for index, (box, points) in enumerate(zip(prediction['boxes'], prediction['points']), 1):
            x1, y1, x2, y2, confidence = box
            axis.add_patch(Rectangle((x1, y1), x2-x1, y2-y1, fill=False,
                                     edgecolor=style['color'], linestyle=style['line'], linewidth=1.8))
            if style['marker'] == 'o':
                axis.scatter(points[:, 0], points[:, 1], s=58, marker='o',
                             facecolors='none', edgecolors=style['color'], linewidths=1.7)
                label_y = y1
                vertical = 'bottom'
                offset = (-8, 6)
            else:
                axis.scatter(points[:, 0], points[:, 1], s=42, marker='x',
                             color=style['color'], linewidths=1.8)
                label_y = y2
                vertical = 'top'
                offset = (6, -10)
            axis.text(x1, label_y, f"{style['short']}{index} {confidence:.3f}",
                      color=style['color'], fontsize=9, va=vertical,
                      bbox={'facecolor': 'black', 'alpha': 0.65, 'pad': 1, 'edgecolor': 'none'},
                      clip_on=True)
            if numbered:
                for point_id, (x, y) in enumerate(points, 1):
                    axis.annotate(str(point_id), (x, y), xytext=offset,
                                  textcoords='offset points', color=style['color'],
                                  fontsize=10, weight='bold',
                                  bbox={'facecolor': 'black', 'alpha': 0.45, 'pad': 0.5,
                                        'edgecolor': 'none'})


def zoom_groups(predictions):
    first = predictions['mtcnn']['boxes']
    second = predictions['retinaface_mnv1_025']['boxes']
    candidates = []
    for i, box_a in enumerate(first):
        for j, box_b in enumerate(second):
            low = np.maximum(box_a[:2], box_b[:2])
            high = np.minimum(box_a[2:4], box_b[2:4])
            intersection = np.prod(np.maximum(high-low, 0))
            area_a = np.prod(box_a[2:4]-box_a[:2])
            area_b = np.prod(box_b[2:4]-box_b[:2])
            overlap = intersection / max(area_a+area_b-intersection, 1e-12)
            if overlap >= PAIR_IOU:
                candidates.append((overlap, i, j))
    used_first = set()
    used_second = set()
    groups = []
    for overlap, i, j in sorted(candidates, reverse=True):
        if i in used_first or j in used_second:
            continue
        groups.append((f'M{i+1} + R{j+1}', [first[i, :4], second[j, :4]]))
        used_first.add(i)
        used_second.add(j)
    for i, box in enumerate(first):
        if i not in used_first:
            groups.append((f'M{i+1} only (no paired RetinaFace box)', [box[:4]]))
    for j, box in enumerate(second):
        if j not in used_second:
            groups.append((f'R{j+1} only (no paired MTCNN box)', [box[:4]]))
    return groups


## 3. แสดงผลทุกรูป: ภาพเต็มและซูมทุกกรอบ
ฟ้า ○ = MTCNN / ส้ม × = RetinaFace; เมื่อตำแหน่งตรงกันจะเห็นกากบาทอยู่ในวงกลม
ภาพ PNG ทุกหน้าถูกบันทึกไว้ใน OUTPUT พร้อม JSON และ CSV


In [ ]:
for image_index, result in enumerate(results, 1):
    image = load_image(result['path'])
    predictions = result['predictions']
    fig, axes = plt.subplots(1, 2, figsize=(16, 9))
    for axis in axes:
        axis.imshow(image)
        axis.axis('off')
    axes[0].set_title('Original image')
    draw_predictions(axes[1], predictions)
    axes[1].set_xlim(0, image.width)
    axes[1].set_ylim(image.height, 0)
    axes[1].set_title('Both models overlaid | boxes + 5 landmarks')
    fig.suptitle(result['image'], fontsize=14)
    fig.legend(handles=legend_handles(), loc='lower center', ncol=2, fontsize=12)
    fig.tight_layout(rect=(0, 0.06, 1, 0.95))
    fig.savefig(OUTPUT / f'{image_index:04d}_full_overlay.png', dpi=140)
    plt.show()
    plt.close(fig)

    groups = zoom_groups(predictions)
    if not groups:
        print('Neither model detected a face at this threshold.')
    for start in range(0, len(groups), 4):
        page = groups[start:start+4]
        fig, axes = plt.subplots(2, 2, figsize=(13, 12))
        for axis in axes.flat:
            axis.axis('off')
        for axis, (label, boxes) in zip(axes.flat, page):
            boxes = np.asarray(boxes)
            low = boxes[:, :2].min(axis=0)
            high = boxes[:, 2:4].max(axis=0)
            padding = np.maximum((high-low) * 0.25, 12)
            low = np.maximum(low-padding, [0, 0])
            high = np.minimum(high+padding, [image.width, image.height])
            axis.imshow(image)
            draw_predictions(axis, predictions, numbered=True)
            axis.set_xlim(low[0], high[0])
            axis.set_ylim(high[1], low[1])
            axis.set_title(label, fontsize=11)
        fig.suptitle(f"{result['image']} | face zoom page {start//4+1}", fontsize=13)
        fig.legend(handles=legend_handles(), loc='lower center', ncol=2, fontsize=12)
        fig.tight_layout(rect=(0, 0.05, 1, 0.95))
        fig.savefig(OUTPUT / f'{image_index:04d}_zoom_{start//4+1:02d}.png', dpi=140)
        plt.show()
        plt.close(fig)
print('Saved overlays, zooms, predictions and summary to:', OUTPUT)
